<a href="https://colab.research.google.com/github/AndresMontesDeOca/vision_computadora_II/blob/VpC2_2026/TP/Couter_Strike.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline de Detección de Objetos en Counter-Strike
Este notebook contiene el pipeline completo para el entrenamiento de modelos de detección de objetos en Counter-Strike. Está dividido en dos grandes fases:
1. **Análisis Exploratorio de Datos (EDA):** Evaluación del dataset, balanceo de clases, dimensiones de imágenes, formatos e identidad cromática en el espacio HSV.
2. **Entrenamiento y Monitoreo (YOLOv11 + W&B):** Configuración de la telemetría de experimentos con Weights & Biases y el entrenamiento del modelo.

### 🛠️ Configuración de Parámetros Globales del Experimento
Define aquí las variables globales y el optimizador que se utilizará en todo el pipeline.

In [1]:
#@title Parámetros de Entrenamiento { run: "auto" }

# Selecciona la versión del modelo preentrenado de YOLO (Soporta YOLOv11 n, s, m, l, x)
yolo_model_version = "yolo11n.pt" #@param ["yolo11n.pt", "yolo11s.pt", "yolo11m.pt", "yolo11l.pt", "yolo11x.pt"] {allow-input: true}

# Selecciona el optimizador ('AdamW' o 'SGD')
optimizer_type = "AdamW" #@param ["AdamW", "SGD"] {allow-input: true}

# Parámetros específicos optimizados para SGD según el estándar de Ultralytics
# sgd_momentum = 0.937 #@param {type:"number"}
# sgd_weight_decay = 0.0005 #@param {type:"number"}

print(f"Configuración seleccionada de forma global:")
print(f" - Modelo: {yolo_model_version}")
print(f" - Optimizador: {optimizer_type}")
if optimizer_type == "SGD":
    print(f"   * Momentum (Momento): {sgd_momentum}")
    print(f"   * Weight Decay (Decaimiento de peso): {sgd_weight_decay}")

Configuración seleccionada de forma global:
 - Modelo: yolo11n.pt
 - Optimizador: AdamW


# 1. Configuración de Librerías e Instalación
Aquí instalamos y cargamos todas las bibliotecas de Python necesarias para trabajar con detección de objetos, visualización y descarga de datos.

In [ ]:
# Instalación de dependencias indispensables (añadimos ultralytics para evitar errores de importación)
!pip install -q roboflow opencv-python matplotlib pyyaml ultralytics

In [ ]:
import os
import yaml
import json
import random
import cv2
import matplotlib.pyplot as plt
from google.colab import userdata
from roboflow import Roboflow

## 2. Carga del Dataset (Data Load)
En esta sección nos conectamos con la API de Roboflow para descargar la versión correspondiente de nuestro dataset de Counter-Strike y cargamos su configuración `data.yaml`.

In [ ]:
import os

# Ruta esperada del dataset
dataset_path = '/content/Counter-Strike-1'

# Verificar si el dataset ya existe en el entorno
if not os.path.exists(dataset_path):
    print("El dataset no fue detectado. Iniciando descarga desde Roboflow...")
    # Obtener la clave API de Roboflow de los secretos de Colab de manera segura
    api_key = userdata.get("ROBOFLOW_API_KEY")
    rf = Roboflow(api_key=api_key)

    # Descarga del dataset en formato YOLOv11
    project = rf.workspace("counter-strike-uovxc").project("counter-strike-h9y80")
    version = project.version(1)
    dataset = version.download("yolov11")
else:
    print(f"El dataset ya existe en '{dataset_path}'. Se omitió la descarga para optimizar tiempo.")

In [ ]:
# Cargar y mostrar la configuración de clases y rutas
with open('/content/Counter-Strike-1/data.yaml', 'r') as file:
    dataset_info = yaml.safe_load(file)

print("Estructura y clases del dataset:")
print(json.dumps(dataset_info, indent=4))

## 3. Funciones Auxiliares para Visualización
Definimos las funciones encargadas de interpretar el formato YOLO (coordenadas de cajas normalizadas) y plasmar visualmente las cajas delimitadoras sobre las imágenes originales.

In [ ]:
def plot_yolo_sample(img_path, lbl_path, class_names):
    """Dibuja cajas YOLO en una imagen usando azul para CT y rojo para Terroristas."""
    if not os.path.exists(img_path):
        print(f"No se encontró la imagen: {img_path}")
        return

    # Cargar y convertir imagen
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image.shape

    # Colores en RGB: Azul para clase 0 (ct), Rojo para clase 1 (t)
    colors = [(0, 0, 255), (255, 0, 0)]

    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id = int(parts[0])
                x_center, y_center, bbox_w, bbox_h = map(float, parts[1:])

                # Convertir coordenadas YOLO normalizadas a píxeles
                x1 = int((x_center - bbox_w / 2) * w)
                y1 = int((y_center - bbox_h / 2) * h)
                x2 = int((x_center + bbox_w / 2) * w)
                y2 = int((y_center + bbox_h / 2) * h)

                # Dibujar rectángulo y texto informativo
                color = colors[cls_id % len(colors)]
                cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)

                label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                cv2.putText(image, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

In [ ]:
def plot_random_samples(images_dir, labels_dir, class_names, num_samples=3):
    """Selecciona de manera aleatoria 'num_samples' imágenes y las renderiza con sus etiquetas."""
    if not os.path.exists(images_dir) or not os.path.exists(labels_dir):
        print("No se encontraron las carpetas de imágenes o etiquetas.")
        return

    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if len(all_images) < num_samples:
        print(f"No hay suficientes imágenes disponibles (Encontradas: {len(all_images)}).")
        return

    selected_images = random.sample(all_images, num_samples)

    for img_name in selected_images:
        lbl_name = img_name.rsplit('.', 1)[0] + '.txt'
        img_path = os.path.join(images_dir, img_name)
        lbl_path = os.path.join(labels_dir, lbl_name)

        print(f"Mostrando muestra aleatoria: {img_name}")
        plot_yolo_sample(img_path, lbl_path, class_names)

## 4. Código de Ejecución de Muestras (Main)
Establecemos las rutas finales y ejecutamos las funciones de visualización utilizando las imágenes y etiquetas de entrenamiento de nuestro dataset.

In [ ]:
# Definición de rutas principales
images_dir = '/content/Counter-Strike-1/train/images'
labels_dir = '/content/Counter-Strike-1/train/labels'
class_names = dataset_info.get('names', ['ct', 't'])

# Ejecutar visualización de 3 imágenes al azar
plot_random_samples(images_dir, labels_dir, class_names, num_samples=3)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Obtener lista de todos los archivos en la carpeta de imágenes
if os.path.exists(images_dir):
    files = os.listdir(images_dir)
    # Extraer la extensión de cada archivo y convertirla a minúsculas
    extensions = [os.path.splitext(f)[1].lower() for f in files if os.path.isfile(os.path.join(images_dir, f))]

    # Crear un contador con Pandas
    df_ext = pd.DataFrame(extensions, columns=['Formato'])
    conteo = df_ext['Formato'].value_counts()

    # Mostrar el conteo en texto
    print("Conteo de formatos de archivos encontrados:")
    print(conteo)

    # Crear gráfico de barras
    plt.figure(figsize=(8, 5))
    conteo.plot(kind='bar', color=['skyblue', 'salmon', 'lightgreen', 'orange'])
    plt.title('Distribución de Formatos de Archivo de las Imágenes')
    plt.xlabel('Formato / Extensión')
    plt.ylabel('Cantidad de Imágenes')
    plt.xticks(rotation=0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
else:
    print(f"No se encontró la ruta de imágenes especificada: {images_dir}")

## 5. Análisis Exploratorio de Datos (EDA) Avanzado

### 5.1 Balanceo de Clases
Calculamos la distribución de las clases (`ct` y `t`) en cada una de las particiones del dataset (Entrenamiento, Validación y Prueba) para verificar su consistencia.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def contar_clases_en_particion(labels_dir_path):
    counts = {0: 0, 1: 0}
    if not os.path.exists(labels_dir_path):
        return counts

    for label_file in os.listdir(labels_dir_path):
        if label_file.endswith('.txt'):
            with open(os.path.join(labels_dir_path, label_file), 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        try:
                            cls_id = int(parts[0])
                            if cls_id in counts:
                                counts[cls_id] += 1
                        except ValueError:
                            continue
    return counts

# Rutas de las carpetas de etiquetas
particiones = {
    'Train': '/content/Counter-Strike-1/train/labels',
    'Valid': '/content/Counter-Strike-1/valid/labels',
    'Test': '/content/Counter-Strike-1/test/labels'
}

resultados = {}
for nombre, ruta in particiones.items():
    resultados[nombre] = contar_clases_en_particion(ruta)

# Crear un DataFrame para facilitar la graficación
df_balance = pd.DataFrame(resultados).rename(index={0: 'Counter-Terrorist (ct)', 1: 'Terrorist (t)'})

print("Frecuencia de clases por partición:")
display(df_balance)

# Graficar los resultados
fig, ax = plt.subplots(figsize=(10, 6))
df_balance.T.plot(kind='bar', ax=ax, color=['royalblue', 'crimson'])
plt.title('Balanceo de Clases (CT vs T) por Partición del Dataset')
plt.xlabel('Partición')
plt.ylabel('Número de Instancias (Cajas Delimitadoras)')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Clases')
plt.show()

### 5.2 Análisis de Resoluciones y Relación de Aspecto (Aspect Ratio)
Analizamos las dimensiones de las imágenes de entrenamiento para conocer su tamaño y verificar si todas comparten la misma resolución y relación de aspecto.

In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt

resoluciones = []

if os.path.exists(images_dir):
    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for img_name in all_images:
        img_path = os.path.join(images_dir, img_name)
        # Leer de forma rápida la imagen
        img = cv2.imread(img_path)
        if img is not None:
            h, w, c = img.shape
            aspect_ratio = w / h
            resoluciones.append({'Ancho': w, 'Alto': h, 'Canales': c, 'Aspect_Ratio': aspect_ratio})

    df_dims = pd.DataFrame(resoluciones)

    # Resumen estadístico
    print("Resumen de las dimensiones y canales de las imágenes:")
    display(df_dims.describe())

    # Crear gráficos de distribución
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Gráfico de dispersión de resoluciones (Ancho vs Alto)
    axes[0].scatter(df_dims['Ancho'], df_dims['Alto'], alpha=0.5, color='purple')
    axes[0].set_title('Dispersión de Resoluciones (Ancho vs Alto)')
    axes[0].set_xlabel('Ancho (píxeles)')
    axes[0].set_ylabel('Alto (píxeles)')
    axes[0].grid(True, linestyle='--', alpha=0.5)

    # Histograma de Relación de Aspecto
    axes[1].hist(df_dims['Aspect_Ratio'], bins=15, color='teal', edgecolor='black', alpha=0.7)
    axes[1].set_title('Distribución de Relación de Aspecto (Aspect Ratio)')
    axes[1].set_xlabel('Relación (Ancho / Alto)')
    axes[1].set_ylabel('Frecuencia')
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()
else:
    print("No se pudo acceder al directorio de imágenes de entrenamiento.")

### 5.3 Análisis de Identidad Cromática por Clase (Aislamiento de Color en HSV)
En esta sección, aislamos de forma matemática la interferencia de los colores del mapa clásico de Counter-Strike (los tonos arena, tierra y sombras del desierto) aplicando filtros de saturación y brillo. De esta forma evaluamos el porcentaje neto real de la presencia de tonos fríos/azules (Hue `[80-140]`) en los uniformes de cada bando.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Inicialización de contadores de píxeles filtrados para cada clase
pixeles_totales_por_clase = {0: 0, 1: 0}
pixeles_azules_por_clase = {0: 0, 1: 0}

if os.path.exists(images_dir) and os.path.exists(labels_dir):
    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    sample_size = min(400, len(all_images))
    selected_images = random.sample(all_images, sample_size)

    for img_name in selected_images:
        lbl_name = img_name.rsplit('.', 1)[0] + '.txt'
        img_path = os.path.join(images_dir, img_name)
        lbl_path = os.path.join(labels_dir, lbl_name)

        if os.path.exists(lbl_path):
            image = cv2.imread(img_path)
            if image is None:
                continue
            h, w, _ = image.shape

            with open(lbl_path, 'r') as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    if cls_id not in [0, 1]:
                        continue

                    x_center, y_center, bbox_w, bbox_h = map(float, parts[1:])
                    x1 = int(max(0, (x_center - bbox_w / 2) * w))
                    y1 = int(max(0, (y_center - bbox_h / 2) * h))
                    x2 = int(min(w, (x_center + bbox_w / 2) * w))
                    y2 = int(min(h, (y_center + bbox_h / 2) * h))

                    crop = image[y1:y2, x1:x2]
                    if crop.size == 0:
                        continue

                    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
                    h_channel = hsv[:, :, 0]
                    s_channel = hsv[:, :, 1]
                    v_channel = hsv[:, :, 2]

                    # Aplicar filtros (Saturación > 60 y Brillo > 30) para evitar grises, negros y el tono de la arena clara
                    color_mask = (s_channel > 60) & (v_channel > 30)
                    valid_hues = h_channel[color_mask]

                    if valid_hues.size > 0:
                        # Filtro de espectro frío/azul (Hue entre 80 y 140)
                        blue_mask = (valid_hues >= 80) & (valid_hues <= 140)
                        blue_pixels_count = np.sum(blue_mask)

                        pixeles_totales_por_clase[cls_id] += valid_hues.size
                        pixeles_azules_por_clase[cls_id] += blue_pixels_count

In [ ]:
porcentajes_azules = {}
for cls_id in [0, 1]:
    totales = pixeles_totales_por_clase[cls_id]
    azules = pixeles_azules_por_clase[cls_id]
    # Usamos class_names que ya está definido en el notebook ['ct', 't']
    porcentajes_azules[class_names[cls_id]] = (azules / totales * 100) if totales > 0 else 0.0

# Crear gráfico comparativo final
plt.figure(figsize=(9, 5))
clases = list(porcentajes_azules.keys())
valores = list(porcentajes_azules.values())

bars = plt.bar(clases, valores, color=['royalblue', 'crimson'], edgecolor='black', width=0.5)
plt.title('Porcentaje Neto de Color Azul/Frío en la Vestimenta por Clase\n(Aislamiento de Entorno Desértico)', fontsize=12, weight='bold')
plt.ylabel('Porcentaje (%) de píxeles azules en el total coloreado', fontsize=11)
plt.ylim(0, max(valores) * 1.2 if max(valores) > 0 else 100)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', weight='bold', fontsize=10)

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 6. Monitoreo de Experimentos (Weights & Biases)
Antes de entrenar cualquier modelo o probar diferentes configuraciones de optimizadores, inicializamos nuestra sesión en **Weights & Biases** de forma segura utilizando la API Key almacenada en los secretos de Colab.

In [ ]:
import os
import wandb
from google.colab import userdata

# Desactivar la integración automática por defecto de Ultralytics para evitar ejecuciones duplicadas
os.environ["WANDB_DISABLED"] = "true"

# Configuración limpia de la nueva corrida de W&B
config_experimento = {
    "model": yolo_model_version,
    "epochs": 25,
    "imgsz": 640,
    "batch": 16,
    "optimizer": optimizer_type,
    "cache": True
}

try:
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print("¡Inicio de sesión en W&B exitoso!")
except Exception as e:
    print(f"Advertencia en login: {e}")

# Inicializar una ejecución completamente nueva
run = wandb.init(
    entity="andresmontesdeoca",
    project="CS2-YOLO-Experiments",
    name=f"clean-{config_experimento['model'].replace('.pt', '')}-{config_experimento['optimizer']}",
    config=config_experimento,
    reinit=True
)

# Definir epoch como el eje X de todas las curvas métricas
wandb.define_metric("epoch")
wandb.define_metric("*", step_metric="epoch")
print(f"Nueva corrida inicializada correctamente. URL de seguimiento:\n{run.url}")

## 7. Entrenamiento del Modelo (YOLOv11)
En esta sección realizaremos la inicialización de los diferentes detectores YOLO, configuraremos el entorno de entrenamiento y ejecutaremos el ajuste fino (fine-tuning) usando la GPU activa. Al estar logueados en W&B, todos los resultados y métricas (pérdidas, mAP, consumo de hardware) se registrarán automáticamente en la nube.

In [ ]:
import torch
import glob
import os
from ultralytics import YOLO

device_to_use = 0 if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de cómputo: {device_to_use}")

yaml_files = glob.glob('/content/**/data.yaml', recursive=True)
if not yaml_files:
    raise FileNotFoundError("No se detectó el archivo data.yaml.")
data_yaml_path = yaml_files[0]

# Instanciar el modelo YOLOv11
model = YOLO(config_experimento["model"])

# --- Implementación de Callbacks Explícitos para control total del log ---
def on_fit_epoch_end(trainer):
    # Extraer métricas de la época actual
    current_epoch = trainer.epoch + 1
    metrics_dict = {"epoch": current_epoch}

    # Pérdidas de entrenamiento
    if hasattr(trainer, 'loss_items') and trainer.loss_items is not None:
        loss_names = ['train/box_loss', 'train/cls_loss', 'train/dfl_loss']
        for i, val in enumerate(trainer.loss_items):
            if i < len(loss_names):
                metrics_dict[loss_names[i]] = float(val)

    # Tasas de Aprendizaje
    if hasattr(trainer, 'optimizer'):
        for i, param_group in enumerate(trainer.optimizer.param_groups):
            metrics_dict[f'lr/group_{i}'] = param_group['lr']

    # Métricas de validación por época
    if hasattr(trainer, 'metrics') and trainer.metrics:
        metrics_dict["val/box_loss"] = float(getattr(trainer, 'validator', {}).loss_items[0]) if hasattr(trainer, 'validator') and hasattr(trainer.validator, 'loss_items') else None
        metrics_dict["val/cls_loss"] = float(trainer.validator.loss_items[1]) if hasattr(trainer, 'validator') and hasattr(trainer.validator, 'loss_items') and len(trainer.validator.loss_items) > 1 else None
        metrics_dict["val/dfl_loss"] = float(trainer.validator.loss_items[2]) if hasattr(trainer, 'validator') and hasattr(trainer.validator, 'loss_items') and len(trainer.validator.loss_items) > 2 else None

        metrics_dict["metrics/precision"] = float(trainer.metrics.get('metrics/precision(B)', 0))
        metrics_dict["metrics/recall"] = float(trainer.metrics.get('metrics/recall(B)', 0))
        metrics_dict["metrics/mAP50"] = float(trainer.metrics.get('metrics/mAP50(B)', 0))
        metrics_dict["metrics/mAP50-95"] = float(trainer.metrics.get('metrics/mAP50-95(B)', 0))

    # Subir el diccionario a W&B
    wandb.log(metrics_dict)
    print(f"\n[W&B Callback] Sincronizando métricas de la Época {current_epoch}...\n")

def on_train_end(trainer):
    print("[W&B Callback] Entrenamiento finalizado localmente. Preparando subida de artefactos...")

# Registrar los callbacks manualmente en el modelo de Ultralytics
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)
model.add_callback("on_train_end", on_train_end)

# Iniciar entrenamiento local (No iniciará de forma automática hasta que ejecutes la celda)
print("Modelo configurado y listo para iniciar el entrenamiento de YOLOv11.")

## 8. Evaluación en el Conjunto de Prueba (Test Set)
Procedemos a cargar los mejores pesos guardados durante el entrenamiento (`best.pt`) y a evaluar formalmente la precisión del modelo utilizando la partición de prueba independiente.

In [ ]:
# Esta celda ha sido unificada y se ejecuta automáticamente dentro de la celda de entrenamiento principal.

## 9. Visualización de Predicciones del Modelo
Seleccionaremos algunas imágenes de la partición de prueba al azar, realizaremos la inferencia con nuestro modelo entrenado y dibujaremos las cajas delimitadoras resultantes junto con su confianza.

In [ ]:
# Esta celda ha sido unificada y se ejecuta automáticamente dentro de la celda de entrenamiento principal.

In [ ]:
if 'run' in globals() and run is not None:
    print(f"🔗 Puedes ver tu corrida en tiempo real en este enlace directo:")
    print(run.get_url())
else:
    print("La sesión de Weights & Biases no parece estar activa en esta ejecución del kernel.")

In [ ]:
# Finalizar explícitamente la corrida para asegurar que todos los datos pendientes se suban y sincronicen a W&B
import wandb
wandb.finish()

In [ ]:
# --- CELDA DE EVALUACIÓN FINAL, SUBIDA DE ARTEFACTOS Y VERIFICACIÓN DE W&B ---
import os
import glob
import wandb
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from google.colab import userdata

# Asegurar dispositivo
device_to_use = 0 if torch.cuda.is_available() else 'cpu'

# Configuración de hiperparámetros con soporte para mayor número de épocas y Early Stopping (Patience = 5)
config_experimento = {
    "model": yolo_model_version,
    "epochs": 100,        # Aumentamos el límite de épocas
    "patience": 5,        # Early Stopping ajustado a 5 épocas consecutivas sin mejora
    "imgsz": 640,
    "batch": 16,
    "optimizer": optimizer_type,
    "cache": True
}

# Agregar parámetros específicos de SGD a la configuración si se seleccionó
extra_train_kwargs = {}
if optimizer_type == "SGD":
    extra_train_kwargs["momentum"] = sgd_momentum
    extra_train_kwargs["weight_decay"] = sgd_weight_decay

# Re-inicializar ejecución en W&B para esta sesión
run = wandb.init(
    entity="andresmontesdeoca",
    project="CS2-YOLO-Experiments",
    name=f"run-{config_experimento['model'].replace('.pt', '')}-{config_experimento['optimizer']}_es",
    config={**config_experimento, **extra_train_kwargs},
    reinit=True
)

# Definir métricas en W&B
wandb.define_metric("epoch")
wandb.define_metric("*", step_metric="epoch")

# Cargar modelo base
model = YOLO(config_experimento["model"])

# callbacks de monitoreo para W&B
def on_fit_epoch_end(trainer):
    current_epoch = trainer.epoch + 1
    metrics_dict = {"epoch": current_epoch}
    if hasattr(trainer, 'loss_items') and trainer.loss_items is not None:
        loss_names = ['train/box_loss', 'train/cls_loss', 'train/dfl_loss']
        for i, val in enumerate(trainer.loss_items):
            if i < len(loss_names):
                metrics_dict[loss_names[i]] = float(val)
    if hasattr(trainer, 'metrics') and trainer.metrics:
        metrics_dict["val/box_loss"] = float(getattr(trainer, 'validator', {}).loss_items[0]) if hasattr(trainer, 'validator') and hasattr(trainer.validator, 'loss_items') else None
        metrics_dict["val/cls_loss"] = float(trainer.validator.loss_items[1]) if hasattr(trainer, 'validator') and hasattr(trainer.validator, 'loss_items') and len(trainer.validator.loss_items) > 1 else None
        metrics_dict["val/dfl_loss"] = float(trainer.validator.loss_items[2]) if hasattr(trainer, 'validator') and hasattr(trainer.validator, 'loss_items') and len(trainer.validator.loss_items) > 2 else None
        metrics_dict["metrics/precision"] = float(trainer.metrics.get('metrics/precision(B)', 0))
        metrics_dict["metrics/recall"] = float(trainer.metrics.get('metrics/recall(B)', 0))
        metrics_dict["metrics/mAP50"] = float(trainer.metrics.get('metrics/mAP50(B)', 0))
        metrics_dict["metrics/mAP50-95"] = float(trainer.metrics.get('metrics/mAP50-95(B)', 0))
    wandb.log(metrics_dict)
    print(f"\n[W&B Callback] Sincronización Época {current_epoch} Completada.\n")

model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

# Buscar archivo data.yaml de forma dinámica
yaml_files = glob.glob('/content/**/data.yaml', recursive=True)
if not yaml_files:
    raise FileNotFoundError("No se detectó el archivo data.yaml para el entrenamiento.")
data_yaml_path = yaml_files[0]

# Ejecutar entrenamiento completo de YOLOv11
results = model.train(
    data=data_yaml_path,
    epochs=config_experimento["epochs"],
    patience=config_experimento["patience"],
    imgsz=config_experimento["imgsz"],
    batch=config_experimento["batch"],
    optimizer=config_experimento["optimizer"],
    cache=config_experimento["cache"],
    device=device_to_use,
    name=f"run_{config_experimento['model'].replace('.pt', '')}_{config_experimento['optimizer']}_es",
    **extra_train_kwargs
)

# OBTENER RUTA PRECISA DE LOS RESULTADOS DEL ENTRENAMIENTO
saved_run_dir = results.save_dir
best_weights = os.path.join(saved_run_dir, "weights", "best.pt")
print(f"\n[+] Pesos finales del mejor modelo localizados en: {best_weights}")

# Evaluación en Validación con el mejor modelo guardado
print("[*] Evaluando mejor modelo en el conjunto de Validación...")
model_best = YOLO(best_weights)
val_results = model_best.val(data=data_yaml_path, split="val")
wandb.log({
    "val/best_mAP50": val_results.results_dict.get("metrics/mAP50(B)", 0.0),
    "val/best_mAP50-95": val_results.results_dict.get("metrics/mAP50-95(B)", 0.0)
})

# Evaluación en Conjunto de Prueba (Test)
print("[*] Evaluando mejor modelo en el conjunto de Prueba (Test)...")
test_results = model_best.val(data=data_yaml_path, split="test")
wandb.log({
    "test/best_mAP50": test_results.results_dict.get("metrics/mAP50(B)", 0.0),
    "test/best_mAP50-95": test_results.results_dict.get("metrics/mAP50-95(B)", 0.0)
})

# Graficación local de resultados
results_csv_path = os.path.join(saved_run_dir, "results.csv")
if os.path.exists(results_csv_path):
    df_results = pd.read_csv(results_csv_path)
    df_results.columns = [c.strip() for c in df_results.columns]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    train_loss = df_results['train/box_loss'] + df_results['train/cls_loss'] + df_results['train/dfl_loss']
    val_loss = df_results['val/box_loss'] + df_results['val/cls_loss'] + df_results['val/dfl_loss']
    epochs = df_results['epoch']

    axes[0].plot(epochs, train_loss, label="Train Total Loss", color="royalblue", linewidth=2.5)
    axes[0].plot(epochs, val_loss, label="Val Total Loss", color="orange", linestyle="--", linewidth=2.5)
    axes[0].set_title("Evolución de Pérdidas (Total Loss vs Época)", fontsize=12, weight="bold")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Pérdida Total")
    axes[0].grid(True, linestyle="--", alpha=0.6)
    axes[0].legend()

    map50_95 = df_results['metrics/mAP50-95(B)']
    map50 = df_results['metrics/mAP50(B)']

    axes[1].plot(epochs, map50_95, label="mAP50-95", color="crimson", linewidth=2.5)
    axes[1].plot(epochs, map50, label="mAP50", color="forestgreen", linestyle=":", linewidth=2)
    axes[1].set_title("Evolución de Métricas de Precisión (mAP vs Época)", fontsize=12, weight="bold")
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("Valor de mAP")
    axes[1].grid(True, linestyle="--", alpha=0.6)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # Guardar gráficos en W&B
    wandb.log({"plots/epoch_vs_loss_chart": wandb.Image(fig)})

# Finalizar formalmente el experimento
print("[*] Finalizando sesión activa de W&B...")
wandb.finish()